# MoleculeNet Interim Analysis

This notebook reproduces the interim analysis for the partial Phase 1 benchmark snapshot pulled from the EC2 node.

By default it reads `summary_partial.csv` in the same directory. To reuse it for the full run later, point `SUMMARY_PATH` at the finished `summary.csv`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SUMMARY_PATH = ROOT / "summary_partial.csv"
OUT_DIR = ROOT / "notebook_outputs"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(SUMMARY_PATH)
df["complete_group"] = df.groupby(["dataset", "split", "model"])["seed"].transform("nunique") == 3
df["group_key"] = df["dataset"] + " | " + df["split"]

complete = df[df["complete_group"]].copy()
agg = complete.groupby(["dataset", "split", "model", "metric_name"], as_index=False).agg(
    metric_mean=("metric_value", "mean"),
    metric_std=("metric_value", "std"),
    seeds=("seed", "nunique"),
)

print(f"rows: {len(df)}")
print(f"complete rows: {len(complete)}")
agg

In [ ]:
if not agg.empty:
    groups = agg[["dataset", "split"]].drop_duplicates().sort_values(["dataset", "split"])
    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 4 * len(groups)), squeeze=False)
    for ax, (_, gs) in zip(axes.flatten(), groups.iterrows()):
        sub = agg[(agg.dataset == gs.dataset) & (agg.split == gs.split)].sort_values("metric_mean", ascending=False)
        ax.bar(sub["model"], sub["metric_mean"], yerr=sub["metric_std"].fillna(0.0), capsize=4)
        ax.set_title(f"{gs.dataset.upper()} | {gs.split} | {sub.metric_name.iloc[0]}")
        ax.set_ylabel(sub.metric_name.iloc[0])
        ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "completed_group_bars.png", dpi=180, bbox_inches="tight")
    plt.show()

In [ ]:
if not agg.empty:
    pivot = agg.pivot_table(index=["dataset", "split"], columns="model", values="metric_mean")
    model_order = sorted(pivot.columns)
    fig, ax = plt.subplots(figsize=(10, max(3, len(pivot) * 0.8)))
    for model in model_order:
        ax.scatter(pivot[model], range(len(pivot)), label=model, s=50)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels([f"{d} | {s}" for d, s in pivot.index])
    ax.set_xlabel("mean metric over completed seeds")
    ax.set_title("Interim MoleculeNet results on completed groups")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "completed_group_scatter.png", dpi=180, bbox_inches="tight")
    plt.show()

pivot